#### **Imports**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

#### **Loading**

In [ ]:
# Load the summary csv file
summary_df = pd.read_csv("../../../data/sequences/moment_2/raw/sequences_summary.csv")
summary_df.head()

#### **Data Analysis**

In [ ]:
# Plot the distribution of rebound types
rebound_counts = summary_df['REBOUND_TYPE'].value_counts()
total = rebound_counts.sum()

def autopct_format(pct):
    count = int(round(pct * total / 100.0))
    return f'{pct:.1f}%\n({count})'

plt.figure(figsize=(4, 4))

wedges, texts, autotexts = plt.pie(
    rebound_counts.values,
    labels=rebound_counts.index,
    autopct=autopct_format,
    startangle=140
)

plt.legend(
    wedges,
    [f"{label}: {count}" for label, count in rebound_counts.items()],
    title="Rebound Type",
    loc="center left",
    bbox_to_anchor=(1, 0.5)
)

plt.title(f'Distribution of Rebound Types (Total = {total})')
plt.axis('equal')

plt.show()

In [ ]:
# Plot the distribution of shot types
shot_counts = summary_df['SHOT_TYPE'].value_counts()
total_shots = shot_counts.sum()
def autopct_format_shots(pct):
    count = int(round(pct * total_shots / 100.0))
    return f'{pct:.1f}%\n({count})'
plt.figure(figsize=(4, 4))
wedges, texts, autotexts = plt.pie(
    shot_counts.values,
    labels=shot_counts.index,
    autopct=autopct_format_shots,
    startangle=140
)
plt.legend(
    wedges,
    [f"{label}: {count}" for label, count in shot_counts.items()],
    title="Shot Type",
    loc="center left",
    bbox_to_anchor=(1, 0.5)
)
plt.title(f'Distribution of Shot Types (Total = {total_shots})')
plt.axis('equal')
plt.show()

In [ ]:
# Compare offensive vs defensive rebounds for different shot types 

# Crosstab counts
rebound_shot = pd.crosstab(summary_df['SHOT_TYPE'], summary_df['REBOUND_TYPE'])
# Convert to percentages within each shot type
rebound_pct = rebound_shot.div(rebound_shot.sum(axis=1), axis=0) * 100
shot_types = rebound_pct.index
offensive_pct = rebound_pct.get('OFFENSIVE', pd.Series(0, index=shot_types))
defensive_pct = rebound_pct.get('DEFENSIVE', pd.Series(0, index=shot_types))

off_color = '#e74c3c'
def_color = '#3498db'

plt.figure(figsize=(14, 6))
x = np.arange(len(shot_types))

# Stacked bars - offensive on bottom, defensive on top
bars_off = plt.bar(x, offensive_pct, color=off_color, alpha=0.8, 
                   label='Offensive %', edgecolor='black', linewidth=0.8)
bars_def = plt.bar(x, defensive_pct, bottom=offensive_pct, color=def_color, 
                   alpha=0.8, label='Defensive %', edgecolor='black', linewidth=0.8)

# Add percentage labels
for i, (o_pct, d_pct) in enumerate(zip(offensive_pct, defensive_pct)):
    # Offensive label (in the middle of offensive section)
    plt.text(i, o_pct/2, f'{o_pct:.1f}%', 
             ha='center', va='center', fontsize=10, fontweight='bold', color='white')
    
    # Defensive label (in the middle of defensive section)
    plt.text(i, o_pct + d_pct/2, f'{d_pct:.1f}%', 
             ha='center', va='center', fontsize=10, fontweight='bold', color='white')

# Add total counts on top of each bar
total_counts = rebound_shot.sum(axis=1)
for i, (shot_type, count) in enumerate(zip(shot_types, total_counts)):
    plt.text(i, 102, f'n={count}', 
             ha='center', va='bottom', fontsize=9, style='italic')

plt.xticks(x, shot_types, fontsize=11)
plt.xlabel('Shot Type', fontsize=12)
plt.ylabel('Percentage (%)', fontsize=12)
plt.title('Rebound Type Distribution by Shot Type (100% Stacked)', 
          fontsize=14, fontweight='bold')
plt.ylim(0, 110)  # Extra space for count labels
plt.legend(fontsize=11, loc='upper right')
plt.axhline(50, color='gray', linestyle='--', linewidth=1, alpha=0.5, label='50% line')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Analyze the distribution of number of frames in the sequences
plt.figure(figsize=(18, 4))
plt.hist(summary_df['NUM_FRAMES'], bins=50, edgecolor='black')
plt.title('Distribution of Number of Frames in Sequences')
plt.xlabel('Number of Frames')
plt.ylabel('Frequency')
plt.grid(axis='y', alpha=0.75)
plt.show()

In [ ]:
# Sequences within +/- N frames of the expected 75-frame length
target = 75
for tol in (1, 10, 20):
    count = ((summary_df['NUM_FRAMES'] - target).abs() <= tol).sum()
    pct = 100 * count / len(summary_df)
    print(f"Within +/-{tol:>2} frames of {target} ({target-tol}-{target+tol}): {count} ({pct:.2f}%)")

In [ ]:
# Check the examples with the least and the most frames
# Sort the summary dataframe by number of frames
sorted_summary = summary_df.sort_values(by='NUM_FRAMES')
print("Example with the least frames:")
sorted_summary.head()

In [ ]:
print("Example with the most frames:")
sorted_summary.tail()

In [ ]:
max_dist = int(summary_df['SHOT_DISTANCE'].max())
bins = np.arange(-0.5, max_dist + 1.5, 1)  # bin edges centered on each integer

plt.figure(figsize=(18, 4))
plt.hist(summary_df['SHOT_DISTANCE'], bins=bins, edgecolor='black')
plt.title('Distribution of Shot Distances in Sequences')
plt.xlabel('Shot Distance (feet)')
plt.ylabel('Frequency')
plt.grid(axis='y', alpha=0.75)
plt.show()

In [ ]:
# Plot comparison of shot distance distributions for offensive vs defensive rebounds
offensive_distances = summary_df[summary_df['REBOUND_TYPE'] == 'OFFENSIVE']['SHOT_DISTANCE']
defensive_distances = summary_df[summary_df['REBOUND_TYPE'] == 'DEFENSIVE']['SHOT_DISTANCE']

# Bin the data on integer feet, so each bin corresponds to exactly one shot distance value
max_dist = int(summary_df['SHOT_DISTANCE'].max())
bins = np.arange(-0.5, max_dist + 1.5, 1)
bin_labels = [f'{int(round((bins[i] + bins[i+1]) / 2))}' for i in range(len(bins) - 1)]

hist_off, _ = np.histogram(offensive_distances, bins=bins)
hist_def, _ = np.histogram(defensive_distances, bins=bins)

# Calculate percentages within each bin
total_per_bin = hist_off + hist_def
pct_off = np.divide(hist_off, total_per_bin, out=np.zeros(len(total_per_bin)), where=total_per_bin != 0) * 100
pct_def = np.divide(hist_def, total_per_bin, out=np.zeros(len(total_per_bin)), where=total_per_bin != 0) * 100

plt.figure(figsize=(18, 6))
x = np.arange(len(bin_labels))

# Stacked bars
plt.bar(x, pct_off, color='#e74c3c', alpha=0.8, label='Offensive %', edgecolor='black', linewidth=0.5)
plt.bar(x, pct_def, bottom=pct_off, color='#3498db', alpha=0.8, label='Defensive %', edgecolor='black', linewidth=0.5)

# Add total case count (n) above each bar, skipping empty bins
for i, total in enumerate(total_per_bin):
    if total > 0:
        plt.text(i, 102, f'{int(total)}', ha='center', va='bottom',
                  fontsize=7, rotation=45, style='italic')

plt.xlabel('Shot Distance (feet)', fontsize=12)
plt.ylabel('Percentage (%)', fontsize=12)
plt.title('Rebound Type Distribution by Shot Distance (100% Stacked)', fontsize=14, fontweight='bold')
plt.xticks(x[::2], [bin_labels[i] for i in range(0, len(bin_labels), 2)], rotation=45, ha='right')
plt.legend(fontsize=11)
plt.grid(axis='y', alpha=0.3)
plt.ylim(0, 115)  # extra headroom for the rotated n= labels
plt.axhline(50, color='gray', linestyle='--', linewidth=1, alpha=0.5)
plt.tight_layout()
plt.show()